# Ultimate Stock Analyzer — Colab Showcase

Notebook de demonstração do estado atual do projeto. O motor financeiro é determinístico; LLM é opcional e nunca é a fonte de verdade para métricas, scores ou ranking.

> Este notebook usa o `main` público. Os dados sintéticos da demonstração não representam recomendações de investimento.


In [ ]:
!git clone -q https://github.com/rodrigorissettoterra/ultimate-stock-analyzer.git
%cd ultimate-stock-analyzer
!python -m pip install -q -e ".[dev]"
print('Projeto instalado.')


## 1. Gate rápido de qualidade


In [ ]:
!ruff check src tests
!pytest -q


## 2. Ranking determinístico

O exemplo abaixo usa empresas sintéticas para mostrar a separação entre qualidade da empresa, atratividade do investimento, timing de entrada e confiança dos dados.


In [ ]:
!python examples/demo_ranking.py


## 3. API v1 dentro do Colab


In [ ]:
import subprocess, time, httpx

server = subprocess.Popen([
    'uvicorn', 'ultimate_stock_analyzer.api.main:app',
    '--host', '127.0.0.1', '--port', '8000'
])
time.sleep(2)
print(httpx.get('http://127.0.0.1:8000/health').json())


In [ ]:
meta = httpx.get('http://127.0.0.1:8000/v1/meta').json()
meta


## 4. Agente conversacional

Sem chave de LLM, o agente usa síntese determinística. Isso é intencional: ausência de evidência não é preenchida por texto inventado.


In [ ]:
response = httpx.post(
    'http://127.0.0.1:8000/v1/agent/query',
    json={'question': 'Explique a evidência mais forte disponível no ranking.'},
    timeout=60,
)
response.json()


## 5. LLM opcional, sem expor segredo

Execute apenas se quiser testar a camada de síntese. A chave não fica gravada no notebook.


In [ ]:
# Opcional
# import getpass, os
# os.environ['USA_LLM_API_KEY'] = getpass.getpass('LLM API key: ')
# os.environ['USA_LLM_MODEL'] = '<model-name>'
# os.environ['USA_LLM_BASE_URL'] = 'https://api.openai.com/v1'


## 6. Estado atual

A arquitetura M0–M20 está implementada, incluindo API, dashboard, agente, backtesting point-in-time, walk-forward e base de produção. O trabalho atual é fechar os gates empíricos/operacionais: cobertura histórica real, contratos setoriais (incluindo SUSEP), backtests multi-regime, calibração OOS, automação dos coletores e validação operacional.

O Colab é ótimo para demonstração e exploração; não substitui o ambiente persistente de produção.


In [ ]:
server.terminate()
server.wait(timeout=10)
print('API encerrada.')
